# Crowding Robustness — Open Images v7 Pipeline
ResNet-34/50/101 · ViT-S/16 · ViT-B/16 · VGG KAGN BN 11v4 · VGG KAGN 11v4

**Adimlar:**
1. GPU kontrol
2. Repo klonla
3. Bagimliliklar
4. Open Images v7 indir
5. SAM2 ile maske iyilestir
6. Isolation gorseller olustur
7. Crowding kompozitleri olustur
8. Modelleri egit (LP-FT)
9. Degerlendir
10. Gorsellestirlmeler

## 0. GPU Kontrolu

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 1. Repo Klonla

In [ ]:
import os, sys

REPO_URL = 'https://github.com/YOUR_USERNAME/crowding-robustness.git'
REPO_DIR = '/content/crowding-robustness'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Dizin:', os.getcwd())

## 2. Bagimliliklar

In [ ]:
import os, sys, site

# Temel bagimliliklar
os.system('pip install -q -r requirements.txt')
os.system('pip install -q timm fvcore fiftyone')

# SAM2
if not os.path.exists('sam2'):
    os.system('git clone https://github.com/facebookresearch/sam2.git')
os.system('pip install -q -e sam2')

# torch-conv-kan (ConvKAN modelleri icin)
if not os.path.exists('torch-conv-kan'):
    os.system('git clone https://github.com/IvanDrokin/torch-conv-kan.git')
os.system('pip install -q -r torch-conv-kan/requirements.txt')

# Kalici path kaydi
with open(os.path.join(site.getsitepackages()[0], 'torch_conv_kan.pth'), 'w') as f:
    f.write('/content/crowding-robustness/torch-conv-kan\n')

print('Kurulum tamamlandi.')

In [ ]:
# Kernel restart gerekebilir (path kaydi icin)
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Restart sonrasi — her session basinda calistir
import os, sys

os.chdir('/content/crowding-robustness')

# torch-conv-kan path'e EKLEME — models/ catismasi yaratir
# Sadece proje kokunu ekle
sys.path = [p for p in sys.path if 'torch-conv-kan' not in p]
if '/content/crowding-robustness' not in sys.path:
    sys.path.insert(0, '/content/crowding-robustness')

# models cache temizle
for mod in list(sys.modules.keys()):
    if 'models' in mod:
        del sys.modules[mod]

# __init__.py garantisi
for d in ['models', 'data', 'data/openimages', 'data/coco']:
    init = f'/content/crowding-robustness/{d}/__init__.py'
    os.makedirs(os.path.dirname(init), exist_ok=True)
    if not os.path.exists(init):
        open(init, 'w').close()

from models.resnet import build_resnet
from models.vit import build_vit
print('Importlar OK')

## 3. Drive Mount (Veri kaydetme/yukleme icin)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/crowding_openimages'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

## 4. Open Images v7 Indir

In [ ]:
import sys, importlib
sys.argv = ['download.py', '--config', 'configs/config_openimages.yaml']

import data.openimages.download as dl
importlib.reload(dl)
dl.main('configs/config_openimages.yaml')

## 5. SAM2 ile Maske Iyilestir

In [ ]:
import sys, importlib
sys.argv = ['refine_masks.py', '--config', 'configs/config_openimages.yaml']

import data.openimages.refine_masks as rm
importlib.reload(rm)
rm.main('configs/config_openimages.yaml')

## 6. Isolation Gorseller Olustur

In [ ]:
import sys, importlib
sys.argv = ['prepare_dataset.py', '--config', 'configs/config_openimages.yaml']

import data.openimages.prepare_dataset as prep
importlib.reload(prep)
prep.main('configs/config_openimages.yaml')

In [ ]:
# Ornek gorseller
import cv2, random, matplotlib.pyplot as plt
from pathlib import Path

dataset_dir = Path('./dataset_openimages')
all_imgs    = list(dataset_dir.rglob('*.png'))
samples     = random.sample(all_imgs, min(8, len(all_imgs)))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, img_path in zip(axes.flatten(), samples):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"{img_path.parts[-2]}\n{img_path.name}", fontsize=7)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Crowding Kompozitleri Olustur

In [ ]:
import sys, importlib
sys.argv = ['build_composites.py', '--config', 'configs/config_openimages.yaml']

import data.openimages.build_composites as bc
importlib.reload(bc)
bc.main('configs/config_openimages.yaml')

In [ ]:
# Ornek kompozitler
import cv2, random, matplotlib.pyplot as plt
from pathlib import Path

comp_dir  = Path('./dataset_openimages/composites/test')
all_comps = list(comp_dir.rglob('*.png'))
samples   = random.sample(all_comps, min(8, len(all_comps)))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, p in zip(axes.flatten(), samples):
    img = cv2.imread(str(p))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    parts = p.parts
    ax.imshow(img)
    ax.set_title(f"{parts[-4]}/{parts[-3]}\n{parts[-2]}", fontsize=6)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 8. Drive'a Kaydet (Ara kayit)

In [ ]:
import shutil, os
print('Dataset kaydediliyor...')
shutil.copytree('./dataset_openimages',
                f'{DRIVE_DIR}/dataset_openimages',
                dirs_exist_ok=True)
print('Tamamlandi.')

## 9. Model Egitimi (LP-FT)

In [ ]:
import sys, importlib
import train as train_module

# Tek model egit
MODEL = 'resnet50'
sys.argv = ['train.py', '--model', MODEL,
            '--config', 'configs/config_openimages.yaml']
importlib.reload(train_module)
train_module.main()

In [ ]:
# Tum modelleri egit
import sys, importlib
import train as train_module

MODELS = ['resnet34', 'resnet50', 'resnet101',
          'vit_s_16', 'vit_b_16',
          'vgg_kagn_bn_11v4', 'vgg_kagn_11v4']

for model_name in MODELS:
    print(f'\n=== {model_name} egitiliyor ===')
    sys.argv = ['train.py', '--model', model_name,
                '--config', 'configs/config_openimages.yaml']
    importlib.reload(train_module)
    train_module.main()

## 10. Degerlendirme

In [ ]:
import sys, importlib
import evaluate as eval_module

sys.argv = ['evaluate.py', '--config', 'configs/config_openimages.yaml']
importlib.reload(eval_module)
eval_module.main()

## 11. Gorsellestirlmeler

In [ ]:
import sys, importlib
import visualize as viz_module

sys.argv = ['visualize.py', '--config', 'configs/config_openimages.yaml']
importlib.reload(viz_module)
viz_module.main()

In [ ]:
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path('results_openimages/figures')
for fig_path in sorted(fig_dir.glob('*.png')):
    print(f'--- {fig_path.name} ---')
    display(Image(str(fig_path)))

## 12. Sonuclari Drive'a Kaydet

In [ ]:
import shutil
shutil.copytree('results_openimages',
                f'{DRIVE_DIR}/results_openimages',
                dirs_exist_ok=True)
print('Sonuclar kaydedildi.')